# Task B judge study: label the Gemini 3.8 Flash segments with candidate judges, exactly as the package does

Task B annotators label the **947 segments** of the 36 Task A windows (Gemini 3.8 Flash segmentation) with one
action each. To choose the labeling judge fairly, every candidate judge must label those segments **exactly as the
real pipeline would**. This notebook does **no segmenting or labeling of its own**: it only runs package commands
(**reasoning-anatomy v1.8.12**) and keeps the books.

For each judge and each creator model:

1. `anatomy check-judge`: one cheap test of the model ID, settings and key (nothing is created if it fails);
2. `anatomy relabel --source-run TaskA/judge_study/g3.8-flash/<MODEL> --allow-unresolved-segments`: the package's own
   command to *label the segments of a finished run with another judge; segmentation is replayed, never re-run*.
   Every segmentation answer of the Task A run is replayed (no segmentation request is answered anew), the package
   checks `segments_identical_to_source`, and its **`05_semantics`** stage labels every segment: same prompt, request
   rendering, earlier-context selection, referenced-setup context, COMMIT and answer-section rules, schema
   validation, retries and receipts as the real pipeline;
3. `anatomy export-annotations`, then the labels are mapped onto the website's segments.

**Why v1.8.12.** v1.8.12 labels the segments of different problems in parallel (v1.8.11 and earlier ran one request
at a time); prompts and labels are unchanged (verified: byte-identical prompts on a finished run). v1.8.11 caps the retries of a label the judge keeps getting wrong at the documented 3 (a wrong
segment ID was retried without limit: 930 calls for one segment). v1.8.10: v1.8.9 did not replay two kinds of saved segmentation answers (the native-ID selections
`native_selection_missing_candidate` / `native_selection_low_cut_density`), so `relabel` stopped (exit 6) and
`resegment` asked them again; they now replay. `anatomy run` and every recorded result are unchanged (study
policy 1.8.8). The Task A runs also have a few unresolved boundaries (34–6,123 tokens per model); v1.8.9 refused to label
such a run at all (exit 3). `--allow-unresolved-segments` labels it with the segments unchanged: segments inside an
unresolved boundary are recorded `TECHNICAL_MISSING` by the package and are **excluded for every judge alike**
(listed in section 4 before anything is paid).

**Everything is read from and written to Drive** (`Reasoning Behavior/TaskB/`). The package runs themselves are written
to the Colab disk while they run (thousands of small files and a database on Drive slow every request down) and are
archived to Drive (`run.tar.gz`) as soon as a model is finished; a model whose `export.json` is on Drive is never run again.

**Resumable.** While a model is being labeled, a checkpoint of its package run is saved to Drive every
`CHECKPOINT_MINUTES` (the package process is paused for a second while its folder is copied, so the copy is consistent).
If Colab disconnects, open the notebook again and run all cells: the run is restored from the checkpoint and the
package continues where the checkpoint left it (at most a few minutes of requests are repeated; nothing is labeled twice).

**Run order:** sections 1–5 in every session (no API cost), then one cell of section 6 per judge (re-running a cell
resumes and never pays twice), then section 7.

## 1. Settings

In [ ]:
RB          = "/content/drive/MyDrive/Reasoning Behavior"     # project folder on Drive
PACKAGE_ZIP = f"{RB}/reasoning-anatomy-v1.8.12.zip"
PACKAGE_VERSION = "1.8.12"
TASKA       = f"{RB}/TaskA"
TASKB       = f"{RB}/TaskB"
KEY         = f"{TASKA}/private/key.csv"                       # window -> trace (private; never published)
OUTPUTS     = f"{RB}/Outputs"                                  # traces.jsonl.gz of every creator model
SOURCE      = f"{TASKA}/judge_study/g3.8-flash"                 # the Gemini 3.8 Flash runs = the segmentation of record
TRACES_JS   = f"{TASKB}/platform/traces.js"                     # the windows and segments annotators label
SITE_TRACES = "https://amirhosseinyari2002.github.io/trace-task-b/traces.js"   # used once to fill TRACES_JS
WORK        = f"{TASKB}/judge_study"                            # one package run per judge x model, costs.csv
JUDGES_OUT  = f"{TASKB}/judges"                                 # <judge>.json for the researcher page
LABELED     = f"{WORK}/labels"                                  # per judge x model: export.json, run.tar.gz
LOCAL_RUNS  = "/content/taskb_runs"                             # package runs while they run (Colab disk; fast)
GEMINI_RPM  = 300            # requests per minute for Gemini judges (package default 120); lower it on 429 errors
CHECKPOINT_MINUTES = 3       # how often a running model is checkpointed to Drive

import os, sys, json, re, csv, glob, gzip, hashlib, datetime, subprocess, bisect, shutil, signal, tarfile, time
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if not IN_COLAB:                                                 # local test with a mirrored folder
    RB = os.environ.get("TASKB_RB", RB)
    TASKA, TASKB, OUTPUTS = f"{RB}/TaskA", f"{RB}/TaskB", f"{RB}/Outputs"
    KEY, SOURCE = f"{TASKA}/private/key.csv", f"{TASKA}/judge_study/g3.8-flash"
    TRACES_JS, WORK, JUDGES_OUT = f"{TASKB}/platform/traces.js", f"{TASKB}/judge_study", f"{TASKB}/judges"
    LABELED, LOCAL_RUNS = f"{WORK}/labels", f"{RB}/_local_runs"

## 2. Setup (once per Colab session): Drive, the package, API keys

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    init = Path("/content/reasoning-anatomy/src/reasoning_anatomy/__init__.py")
    if not (init.exists() and f'__version__ = "{PACKAGE_VERSION}"' in init.read_text()):
        subprocess.run(["rm", "-rf", "/content/reasoning-anatomy"], check=True)
        subprocess.run(["unzip", "-q", "-o", PACKAGE_ZIP, "-d", "/content/"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", "/content/reasoning-anatomy[research,api]"], check=True)
    from google.colab import userdata
    for k in ("GEMINI_API_KEY", "ANTHROPIC_API_KEY"):          # key icon -> add the secret
        try: os.environ[k] = userdata.get(k)
        except Exception: print(f"no {k} secret: judges of that provider stop at their check until you add it")
    if "/content/reasoning-anatomy/src" not in sys.path: sys.path.insert(0, "/content/reasoning-anatomy/src")
installed = subprocess.run([sys.executable, "-c", "import reasoning_anatomy as r; print(r.__version__)"], capture_output=True, text=True).stdout.strip()
print("reasoning-anatomy:", installed or "not installed")
if IN_COLAB: assert installed == PACKAGE_VERSION, f"reasoning-anatomy {PACKAGE_VERSION} is needed (found {installed or 'none'})"
for d in (f"{TASKB}/platform", WORK, LABELED, JUDGES_OUT, f"{TASKB}/annotations", f"{TASKB}/results"): os.makedirs(d, exist_ok=True)

## 3. Inputs: the windows and segments (Drive), the key, the traces, the Gemini 3.8 Flash runs

`TaskB/platform/traces.js` is the exact file the annotators' website uses (copied once from the site, then always
read from Drive). Its text and segment hashes are checked, and every window's text is checked against the trace
it was cut from.

In [ ]:
def read_traces_js(path):
    raw = open(path, encoding="utf-8").read()
    k = raw.index("window.TRACES = ") + len("window.TRACES = ")
    src = json.loads(re.search(r"window\.TASKB_SOURCE = (\{.*?\});", raw).group(1))
    return json.loads(raw[k: raw.rindex(";")]), src
if not os.path.exists(TRACES_JS):
    import urllib.request
    open(TRACES_JS, "w", encoding="utf-8").write(urllib.request.urlopen(SITE_TRACES, timeout=60).read().decode("utf-8"))
    print("copied the website's traces.js to", TRACES_JS)
WINDOWS, SOURCE_INFO = read_traces_js(TRACES_JS)
WIN = {w["id"]: w for w in WINDOWS}
for w in WINDOWS:
    assert hashlib.sha256(w["text"].encode("utf-8")).hexdigest() == w["sha256"], w["id"] + ": text hash"
    assert hashlib.sha256(json.dumps(w["segs"]).encode()).hexdigest() == w["seg_sha256"], w["id"] + ": segmentation hash"
N_SEG = sum(len(w["segs"]) for w in WINDOWS)
print(f"{len(WINDOWS)} windows, {N_SEG} segments; segmentation {SOURCE_INFO['judge']}")

KEYROWS = {r["item_id"]: r for r in csv.DictReader(open(KEY, encoding="utf-8")) if str(r["practice"]).lower() != "true"}
missing = sorted(set(WIN) - set(KEYROWS))
assert not missing, f"windows not in key.csv: {missing}"
MODELS = sorted({KEYROWS[i]["model_code"] for i in WIN})

def load_traces(codes):
    """code -> trace_id, completion_text, reasoning (the text the windows were cut from), as the Task A kit reads them."""
    want, found = set(codes), {}
    for path in sorted(glob.glob(os.path.join(OUTPUTS, "*", "traces.jsonl.gz"))):
        prefixes = {c.split("-")[0] for c in want - set(found)}
        if not prefixes: break
        with gzip.open(path, "rt", encoding="utf-8") as f:
            first = True
            for line in f:
                row = json.loads(line)
                if first:
                    first = False
                    if row["code"].split("-")[0] not in prefixes: break
                if row["code"] in want: found[row["code"]] = {k: row[k] for k in ("trace_id", "completion_text", "reasoning")}
    return found
TRACE = load_traces([KEYROWS[i]["code"] for i in WIN])
for i, w in WIN.items():
    k, t = KEYROWS[i], TRACE[KEYROWS[i]["code"]]
    comp, reas = t["completion_text"], t["reasoning"]
    assert comp.count(reas) == 1, i + ": reasoning is not a unique part of completion_text"
    base = comp.index(reas) + int(k["char_start"])
    assert comp[base: base + len(w["text"])] == w["text"], i + ": the website window text differs from the trace"
    w["_base"], w["_trace_id"], w["_model"] = base, t["trace_id"], k["model_code"]
print("every window matches its trace exactly; models:", MODELS)
for m in MODELS:
    assert os.path.exists(f"{SOURCE}/{m}/resolved_config.json"), f"no Gemini 3.8 Flash run for {m} in {SOURCE}"

## 4. Package helpers and the checks before anything is paid (free)

* **Replay audit:** with the installed package's own rules (`cacheable_purpose`, `replayable_response`), every
  segmentation answer saved in each Task A run must be replayable, so `relabel` reproduces the segmentation without
  a new segmentation answer. It also flags two saved requests with the identical request but different answers
  (the package would replay the first; `relabel` then reports the segments differ).
* **Segmentation of record:** the Task A runs' segments are mapped onto the 947 website segments (every one must be
  covered).
* **Unresolved boundaries:** website segments that lie in an unresolved boundary of the Task A run (the package
  labels them `TECHNICAL_MISSING`) are listed; they are excluded for every judge (`TaskB/judge_study/excluded.json`).

In [ ]:
def anatomy(*args):
    """Run the package's command line; returns (exit code, short message, parsed JSON result)."""
    print("  $ anatomy " + " ".join(map(str, args)), flush=True)
    p = subprocess.run([sys.executable, "-m", "reasoning_anatomy", *map(str, args)], capture_output=True, text=True)
    lines = (p.stdout or "").strip().splitlines()
    last = lines[-1] if lines else (p.stderr or "").strip()[-400:]
    try: parsed = json.loads(last)
    except ValueError: parsed = None
    short = (parsed.get("error") or parsed.get("status")) if isinstance(parsed, dict) else last
    return p.returncode, str(short)[:400], parsed

ANATOMY_CMD = [sys.executable, "-m", "reasoning_anatomy"]

def _short(parsed, last):
    return str((parsed.get("error") or parsed.get("status")) if isinstance(parsed, dict) else last)[:400]

def checkpoint(rd, ckpt, proc=None):
    """Save a consistent copy of the package run `rd` to Drive (`ckpt`). A running process is paused while copying."""
    rd, ckpt = Path(rd), Path(ckpt)
    snap = Path(LOCAL_RUNS) / "_snapshots" / f"{rd.parent.name}__{rd.name}"
    shutil.rmtree(snap, ignore_errors=True); snap.parent.mkdir(parents=True, exist_ok=True)
    if proc is not None and proc.poll() is None: os.killpg(proc.pid, signal.SIGSTOP)
    try:   # temporary files, the SQLite shared-memory index (rebuilt on open) and the lock file are not copied
        shutil.copytree(rd, snap, ignore=shutil.ignore_patterns(".*", "*.sqlite3-shm", "run.lock"))
    finally:
        if proc is not None and proc.poll() is None: os.killpg(proc.pid, signal.SIGCONT)
    local = snap.with_suffix(".tar.gz")
    with tarfile.open(local, "w:gz") as t: t.add(snap, arcname=rd.name)
    ckpt.parent.mkdir(parents=True, exist_ok=True)
    part = ckpt.with_name(ckpt.name + ".part")
    shutil.copyfile(local, part); os.replace(part, ckpt)
    shutil.rmtree(snap, ignore_errors=True); local.unlink(missing_ok=True)

def run_resumable(args, rd, ckpt):
    """`anatomy <args>` for the run folder `rd`, checkpointed to Drive (`ckpt`) and restored from it after a disconnect."""
    rd, ckpt = Path(rd), Path(ckpt)
    if not (rd / "run.sqlite3").exists() and ckpt.exists():
        shutil.rmtree(rd, ignore_errors=True); rd.parent.mkdir(parents=True, exist_ok=True)
        with tarfile.open(ckpt) as t:
            t.extractall(rd.parent, **({"filter": "data"} if hasattr(tarfile, "data_filter") else {}))
        print(f"  restored from the Drive checkpoint of {datetime.datetime.fromtimestamp(ckpt.stat().st_mtime):%Y-%m-%d %H:%M}", flush=True)
    rd.parent.mkdir(parents=True, exist_ok=True)
    print("  $ anatomy " + " ".join(map(str, args)), flush=True)
    out_path, err_path = rd.parent / f"{rd.name}.stdout", rd.parent / f"{rd.name}.stderr"
    with open(out_path, "w") as out, open(err_path, "a") as err:
        proc = subprocess.Popen([*ANATOMY_CMD, *map(str, args)], stdout=out, stderr=err, text=True, start_new_session=True)
        last = time.time()
        try:
            while proc.poll() is None:
                time.sleep(2)
                if time.time() - last >= CHECKPOINT_MINUTES * 60 and (rd / "run.sqlite3").exists():
                    checkpoint(rd, ckpt, proc); last = time.time()
        except KeyboardInterrupt:          # Colab "stop": let the package pause cleanly, then keep a checkpoint
            os.killpg(proc.pid, signal.SIGINT); proc.wait()
            if (rd / "run.sqlite3").exists(): checkpoint(rd, ckpt)
            raise
    lines = out_path.read_text().strip().splitlines()
    last_line = lines[-1] if lines else err_path.read_text().strip()[-400:]
    try: parsed = json.loads(last_line)
    except ValueError: parsed = None
    if proc.returncode != 0 and (rd / "run.sqlite3").exists(): checkpoint(rd, ckpt)
    return proc.returncode, _short(parsed, last_line), parsed

_AUDIT = r"""
import json, sys
from collections import Counter
from pathlib import Path
from reasoning_anatomy.boundary_requests import cache_key
from reasoning_anatomy.reuse import cacheable_purpose, replayable_response
rd = Path(sys.argv[1]); c = Counter(); not_replayable = Counter(); seen = {}
for ip in sorted((rd / "stages/04_atomicity/requests").glob("*.intent.json")):
    intent = json.loads(ip.read_text()); rp = ip.with_name(ip.name.replace(".intent", ".receipt"))
    if not rp.exists(): c["paid_without_receipt"] += 1; continue
    r = json.loads(rp.read_text())["response"]
    if not replayable_response(r): c["transport_failures"] += 1; continue
    if not cacheable_purpose(intent.get("purpose")): not_replayable[intent.get("purpose")] += 1; continue
    c["replayable_answers"] += 1
    k = cache_key(intent, r["model_version"]); answer = json.dumps(r.get("raw_text") or r.get("parsed"), sort_keys=True)
    if k in seen and seen[k] != answer: c["same_request_different_answer"] += 1
    seen.setdefault(k, answer)
q = rd / "stages/04_atomicity/quality.json"; q = json.loads(q.read_text()) if q.exists() else {}
print(json.dumps({"counts": c, "not_replayable": not_replayable, "all_boundaries_resolved": q.get("all_boundaries_resolved"),
                  "unresolved_tokens": q.get("unresolved_tokens"), "blocking_issue_counts": q.get("blocking_issue_counts")}))
"""
def replay_audit(rd):
    p = subprocess.run([sys.executable, "-c", _AUDIT, str(rd)], capture_output=True, text=True)
    if p.returncode: raise SystemExit(f"audit of {rd} failed: {p.stderr[-400:]}")
    return json.loads(p.stdout)

def append_cost(row):
    """One row per judge x model x stage (replaced when recorded again: usage is cumulative)."""
    path = Path(WORK) / "costs.csv"
    rows = list(csv.DictReader(open(path))) if path.exists() else []
    if row["stage"] != "check-judge":
        rows = [r for r in rows if not (r["judge"] == row["judge"] and r["model_code"] == row["model_code"] and r["stage"] == row["stage"])]
    rows.append({k: str(v) for k, v in row.items()})
    fields = list(dict.fromkeys(list(row) + [k for r in rows for k in r]))
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, restval=""); w.writeheader(); w.writerows(rows)
    os.replace(tmp, path)
now = lambda: datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")

AUDIT, READY = {}, []
for m in MODELS:
    a = AUDIT[m] = replay_audit(f"{SOURCE}/{m}")
    ok = not a["not_replayable"]
    if ok: READY.append(m)
    print(f"{m}: {dict(a['counts'])}; not replayable {dict(a['not_replayable']) or 'none'}; "
          f"complete={a['all_boundaries_resolved']} unresolved tokens={a['unresolved_tokens']} -> {'ready' if ok else 'NOT READY'}")
if len(READY) < len(MODELS):
    print("\nNOT READY models have saved answers this package version cannot replay: check that v1.8.10 is installed (section 2).")
if any(a["counts"].get("same_request_different_answer") for a in AUDIT.values()):
    print("\nnote: a run holds the identical request twice with different answers; relabel will report whether its segments differ.")

In [ ]:
def export_of(run_dir, out):
    if not os.path.exists(out):
        os.makedirs(os.path.dirname(out), exist_ok=True)
        code, short, _ = anatomy("export-annotations", "--run-dir", run_dir, "--output", out)
        if code != 0 or not os.path.exists(out): raise SystemExit(f"export of {run_dir} failed: {short}")
    return out

def map_export(export_path, items):
    """website segments of `items` -> list of the package segments that make them up (dicts from the export)."""
    doc = json.load(open(export_path, encoding="utf-8"))
    if doc.get("format") != "saae-human-alignment-v1" or doc.get("kind") != "prediction":
        raise SystemExit(f"{export_path}: not an `anatomy export-annotations` prediction file")
    by_tid = {t["trace_id"]: t for t in doc["traces"]}
    out = {}
    for i in items:
        w = WIN[i]; t = by_tid.get(w["_trace_id"])
        if t is None: raise SystemExit(f"{export_path}: trace of window {i} is missing")
        comp = TRACE[KEYROWS[i]["code"]]["completion_text"]
        if t["text_sha256"] != hashlib.sha256(comp.encode("utf-8")).hexdigest():
            raise SystemExit(f"{i}: the run's trace text differs from Outputs")
        base, end = w["_base"], w["_base"] + len(w["text"])
        starts = [base + a for _, a, _ in w["segs"]]
        parts = [[] for _ in w["segs"]]
        for s in t["segments"]:
            if s["end"] <= base or s["start"] >= end: continue                 # outside the window
            txt = comp[s["start"]: s["end"]]
            core = s["start"] + (len(txt) - len(txt.lstrip()))
            if not txt.strip() or core >= end: continue                         # whitespace only, or starts after it
            k = bisect.bisect_right(starts, max(core, base)) - 1
            parts[k].append(s)
        empty = [k for k, p in enumerate(parts) if not p]
        if empty: raise SystemExit(f"{i}: website segment(s) {empty} contain no package segment: the runs are not the segmentation of record")
        out[i] = parts
    return out

LABELS = ("CONSTRUCT", "INSPECT", "REVISE", "REDIRECT", "NONACTION")
bounds = lambda parts: [(s["start"], s["end"]) for s in parts]
SOURCE_PARTS, EXCLUDED, one = {}, {}, 0
for m in MODELS:
    items = [i for i in WIN if WIN[i]["_model"] == m]
    mp = map_export(export_of(f"{SOURCE}/{m}", f"{WORK}/source_exports/{m}.json"), items)
    for i, parts in mp.items():
        SOURCE_PARTS[i] = [bounds(p) for p in parts]
        for k, p in enumerate(parts):
            one += len(p) == 1
            if any(not s.get("boundary_resolved", True) for s in p): EXCLUDED.setdefault(i, {})[str(k)] = "unresolved_boundary"
n_ex = sum(len(v) for v in EXCLUDED.values())
print(f"Task A runs map onto all {N_SEG} website segments ({one} one-to-one).")
print(f"in an unresolved boundary (TECHNICAL_MISSING for every judge, excluded): {n_ex} of {N_SEG} website segments")
order = [w["id"] for w in WINDOWS]
for i in sorted(EXCLUDED, key=order.index):
    print(f"  {WIN[i]['label']} ({i}, {WIN[i]['_model']}): segments {sorted(int(k) + 1 for k in EXCLUDED[i])}")
json.dump({"excluded": EXCLUDED, "audit": AUDIT, "created": now()}, open(f"{WORK}/excluded.json", "w"), indent=1)

## 5. Candidate judges (edit here) and the package calls

Same format as the Task A judge study (`settings` = the package's AnnotationConfig fields; `price` = USD per 1M
tokens, output includes thinking). Each judge's settings and prices are written **once** to
`TaskB/judge_study/<judge>/judge-config.json`; a later edit of a judge that has already run is refused (give it a
new name instead), so a resumed run is never mixed. Check IDs and prices on your pricing pages first.
Claude judges get `max_input_tokens_per_problem` = 4,000,000 (package default 2,000,000): Claude counts more input
tokens per prompt, and the two longest NEM traces exhausted the default, leaving their last segments unlabeled.
The cap only limits spending; no prompt or label changes (the Gemini judges stayed far below it).

In [ ]:
JUDGES = {
  "g3.8-flash":             {"provider": "gemini", "model": "gemini-3.8-flash", "price": [0.75, 3.75],
                             "settings": {"temperature": None, "gemini_thinking_level": "low", "gemini_thinking_budget": None, "max_output_tokens": 4096},
                             "note": "the segmentation judge; introductory price to 2026-12-31, then 1.50 / 7.50"},
  "g3.5-flash-lite":        {"provider": "gemini", "model": "gemini-3.5-flash-lite", "price": [0.30, 2.50],
                             "settings": {"temperature": None, "gemini_thinking_level": "minimal", "gemini_thinking_budget": None}},
  "g3.5-flash-lite-think":  {"provider": "gemini", "model": "gemini-3.5-flash-lite", "price": [0.30, 2.50],
                             "settings": {"temperature": None, "gemini_thinking_level": "low", "gemini_thinking_budget": None, "max_output_tokens": 4096}},
  "g3.1-flash-lite":        {"provider": "gemini", "model": "gemini-3.1-flash-lite", "price": [0.25, 1.50],
                             "settings": {"temperature": None, "gemini_thinking_level": "minimal", "gemini_thinking_budget": None}},
  "g2.5-flash-lite":        {"provider": "gemini", "model": "gemini-2.5-flash-lite", "price": [0.10, 0.40],
                             "settings": {"temperature": None, "gemini_thinking_level": None, "gemini_thinking_budget": 0}},
  "g4-hydrogen":            {"provider": "gemini", "model": "gemini-4-hydrogen", "price": [1.50, 7.50],
                             "settings": {"temperature": None, "gemini_thinking_level": "low", "gemini_thinking_budget": None, "max_output_tokens": 4096}},
  "claude-haiku-5.5":       {"provider": "claude", "model": "claude-haiku-5-5", "price": [0.10, 0.50],
                             "settings": {"temperature": None, "gemini_thinking_level": None, "gemini_thinking_budget": None, "claude_thinking": "disabled", "reasoning_effort": "low", "max_input_tokens_per_problem": 4000000}},
  "claude-haiku-5.5-think": {"provider": "claude", "model": "claude-haiku-5-5", "price": [0.10, 0.50],
                             "settings": {"temperature": None, "gemini_thinking_level": None, "gemini_thinking_budget": None, "claude_thinking": "adaptive", "reasoning_effort": "low", "max_output_tokens": 8192, "max_input_tokens_per_problem": 4000000}},
}

def receipts_usage(run_dir, stages=("05_semantics",)):
    """Exact usage from the package's per-request receipts (as the Task A kit computes it)."""
    out = {}
    for stage in stages:
        files = glob.glob(str(Path(run_dir) / "stages" / stage / "requests" / "*.receipt.json"))
        n = pin = pout = think = cread = cwrite = 0
        for f in files:
            try: d = json.load(open(f))
            except (ValueError, OSError): continue
            for r in d.get("transport_attempts") or ([d["response"]] if "response" in d else []):
                meta = r.get("response_metadata") or {}
                p = r.get("prompt_tokens") or meta.get("prompt_token_count") or 0
                o = r.get("output_tokens") or meta.get("candidates_token_count") or 0
                t = meta.get("thoughts_token_count") or 0
                total = meta.get("total_token_count")
                usage = meta.get("usage") or {}
                cread += int(usage.get("cache_read_input_tokens") or 0); cwrite += int(usage.get("cache_creation_input_tokens") or 0)
                n += 1; pin += p; pout += (total - p) if total else (o + t); think += t
        out[stage] = {"requests": n, "input_tokens": pin, "output_tokens": pout, "thinking_tokens": think,
                      "cache_read_tokens": cread, "cache_write_tokens": cwrite}
    return out
def cost(u, price):
    read, write = u.get("cache_read_tokens", 0), u.get("cache_write_tokens", 0)
    billed_in = u["input_tokens"] - read - write + 0.1 * read + 1.25 * write
    return round((billed_in * price[0] + u["output_tokens"] * price[1]) / 1e6, 6)
print("judges:", list(JUDGES))

## 6. Run the judges: one cell per judge

Each cell: `check-judge` (a few cents; a wrong model ID, setting or key stops here and creates nothing), then for each
creator model `anatomy relabel` of the Task A run (with `--allow-unresolved-segments` when that run has unresolved
boundaries) and `anatomy export-annotations`, the cost from the receipts, and finally `TaskB/judges/<judge>.json`.
The package labels **every segment of the 36 traces** (not only the window segments), as the real pipeline would;
only the window segments are compared with the annotators. **Resume:** after an interruption or a Colab disconnect, run all
cells again (sections 1–5 cost nothing) and then the same judge cell: finished models (their `export.json` on Drive)
are skipped, and a model that was running continues from its last checkpoint (`checkpoint.tar.gz`, at most
`CHECKPOINT_MINUTES` old). A model is never labeled twice: the package keeps every answered request.

In [ ]:
def website_labels(parts, excluded=None):
    """one label per website segment; None when not comparable."""
    if excluded: return None, excluded
    acts = [s.get("action") for s in parts]
    if len(set(acts)) == 1 and acts[0] in LABELS: return acts[0], "ok"
    if any(a is None for a in acts): return None, "missing"
    if "COMMIT" in acts: return None, "commit"
    return None, "parts_disagree"

def run_judge(name, models=None):
    J = JUDGES[name]; price = J["price"]; jdir = Path(LABELED) / name; jdir.mkdir(parents=True, exist_ok=True)
    overrides = dict(J["settings"], input_usd_per_million=price[0], output_usd_per_million=price[1],
                     pricing_snapshot=f"TaskB judge study ({name}); verify on the provider's pricing page")
    if J["provider"] == "gemini": overrides["max_requests_per_minute"] = GEMINI_RPM   # pace only; labels unaffected
    cfg = jdir / "judge-config.json"
    if cfg.exists():
        if json.loads(cfg.read_text()) != overrides:
            raise SystemExit(f"{name} already ran with other settings ({cfg}). To re-run it with the new settings, delete "
                             f"the folder {jdir} and {Path(JUDGES_OUT) / (name + '.json')} on Drive, then run this cell again; "
                             f"or give the new settings a new judge name.")
    else:
        cfg.write_text(json.dumps(overrides, indent=1))
    todo = [m for m in (models or MODELS) if m in READY]
    if len(todo) < len(models or MODELS): print("not ready (section 4):", [m for m in (models or MODELS) if m not in READY])
    check = jdir / "check-judge.json"
    if not check.exists() or not json.loads(check.read_text()).get("passed"):
        code, short, report = anatomy("check-judge", "--judge-model", J["model"], "--judge-config", cfg)
        if report: check.write_text(json.dumps(report, indent=1))
        billed = [t for t in (report or {}).get("tasks", {}).values() if t.get("ok")]
        u = {"requests": len(billed), "input_tokens": sum(t.get("input_tokens") or 0 for t in billed),
             "output_tokens": sum(t.get("output_tokens") or 0 for t in billed), "thinking_tokens": 0, "cache_read_tokens": 0, "cache_write_tokens": 0}
        append_cost({"time": now(), "judge": name, "model_id": J["model"], "model_code": "-", "stage": "check-judge", **u,
                     "price_in": price[0], "price_out": price[1], "cost_usd": cost(u, price), "run_dir": ""})
        if code != 0:
            raise SystemExit(f"judge check failed: {short} (details in {check}); nothing was created")
    results = {}
    for m in todo:
        out = jdir / m
        if (out / "export.json").exists():
            results[m] = "done (on Drive)"; continue
        rd = Path(LOCAL_RUNS) / name / m
        print(f"\n[{name} / {m}]", flush=True)
        extra = [] if AUDIT[m]["all_boundaries_resolved"] else ["--allow-unresolved-segments"]
        ckpt = out / "checkpoint.tar.gz"
        code, short, res = run_resumable(["relabel", "--source-run", f"{SOURCE}/{m}", "--run-dir", rd, "--annotator", J["provider"],
                                          "--annotator-model", J["model"], "--annotation-config", cfg, *extra], rd, ckpt)
        if code != 0 or not isinstance(res, dict) or res.get("status") != "completed":
            print(f"  not finished (exit {code}): {short}\n  checkpoint saved; run this cell again (also after a disconnect) to continue."); results[m] = f"not finished: {short}"; continue
        if res.get("segments_identical_to_source") is not True:
            raise SystemExit(f"{m}: the relabel run's segments differ from the Task A Gemini 3.8 Flash run")
        code, short, _ = anatomy("export-annotations", "--run-dir", rd, "--output", rd / "export.json")
        if code != 0: results[m] = f"export failed: {short}"; continue
        u = receipts_usage(rd)["05_semantics"]; c = cost(u, price)
        out.mkdir(parents=True, exist_ok=True)
        shutil.make_archive(str(out / "run"), "gztar", root_dir=rd.parent, base_dir=m)    # the whole package run
        shutil.copyfile(rd / "export.json", out / "export.tmp"); os.replace(out / "export.tmp", out / "export.json")
        (out / "checkpoint.tar.gz").unlink(missing_ok=True)                              # the finished run is in run.tar.gz
        append_cost({"time": now(), "judge": name, "model_id": J["model"], "model_code": m, "stage": "05_semantics", **u,
                     "price_in": price[0], "price_out": price[1], "cost_usd": c, "run_dir": str(out / "run.tar.gz")})
        results[m] = f"labeled, ${c:.4f}" + ("" if not extra else f" ({res.get('unresolved_tokens')} unresolved tokens: TECHNICAL_MISSING)")
    print(json.dumps(results, indent=1))
    if all((jdir / m / "export.json").exists() for m in MODELS):
        return write_judge(name)
    print("judge file not written yet: some models are not finished (run the cell again)")

def write_judge(name):
    J = JUDGES[name]; jdir = Path(LABELED) / name
    labels, actions, parts_n, reasons = {}, {}, {}, {}
    for m in MODELS:
        mp = map_export(jdir / m / "export.json", [i for i in WIN if WIN[i]["_model"] == m])
        for i, parts in mp.items():
            if [bounds(p) for p in parts] != SOURCE_PARTS[i]:
                raise SystemExit(f"{i}: the judge's segments differ from the Task A run")
            res = [website_labels(p, EXCLUDED.get(i, {}).get(str(k))) for k, p in enumerate(parts)]
            labels[i] = [r[0] for r in res]
            actions[i] = [[s.get("action") for s in p] for p in parts]
            parts_n[i] = [len(p) for p in parts]
            for _, why in res: reasons[why] = reasons.get(why, 0) + 1
    costs = [r for r in csv.DictReader(open(Path(WORK) / "costs.csv")) if r["judge"] == name]
    run = json.load(open(jdir / MODELS[0] / "export.json"))["run"]
    out = {"format": "trace-task-b-judge/v2", "judge": name, "provider": J["provider"], "model": J["model"],
           "returned_model_version": run.get("returned_model_version"), "settings": J["settings"], "price": J["price"],
           "package_version": PACKAGE_VERSION,
           "protocol": "anatomy relabel of TaskA/judge_study/g3.8-flash/<MODEL> (every segmentation answer replayed, "
                       "segments identical; --allow-unresolved-segments where the run has unresolved boundaries), the package's "
                       "own 05_semantics stage, anatomy export-annotations; labels mapped onto the website segments; segments in "
                       "an unresolved boundary are TECHNICAL_MISSING and excluded for every judge",
           "segmentation": SOURCE_INFO, "seg_sha256": {i: WIN[i]["seg_sha256"] for i in WIN},
           "unresolved_tokens": {m: AUDIT[m]["unresolved_tokens"] for m in MODELS},
           "labels": labels, "package_actions": actions, "package_segments_per_website_segment": parts_n,
           "comparable": reasons, "cost_usd": round(sum(float(r["cost_usd"]) for r in costs), 4),
           "requests": sum(int(float(r["requests"] or 0)) for r in costs), "created": now()}
    path = Path(JUDGES_OUT) / f"{name}.json"
    json.dump(out, open(path, "w"), ensure_ascii=False)
    dist = {}
    for v in labels.values():
        for x in v: dist[x] = dist.get(x, 0) + 1
    print(f"wrote {path}: {reasons}; labels {dist}; cost so far ${out['cost_usd']}")
    return out

In [ ]:
run_judge("g3.8-flash")

In [ ]:
run_judge("g3.5-flash-lite")

In [ ]:
run_judge("claude-haiku-5.5")

In [ ]:
run_judge("claude-haiku-5.5-think")

Optional judges:

In [ ]:
# run_judge("g2.5-flash-lite")
# run_judge("g3.5-flash-lite-think")
# run_judge("g3.1-flash-lite")
# run_judge("g4-hydrogen")

## 7. Costs, label distributions, judge–judge agreement

The decision is made on the **researcher page**: Agreement → Judges → load `TaskB/judges/*.json` and
`TaskA/private/key.csv`. Download *all metrics (JSON)* there and save it in `TaskB/results/`.

In [ ]:
import pandas as pd
c = pd.read_csv(f"{WORK}/costs.csv")
display(c.groupby(["judge", "stage"])[["requests", "input_tokens", "output_tokens", "cost_usd"]].sum())
J = {p.stem: json.load(open(p)) for p in sorted(Path(JUDGES_OUT).glob("*.json"))}
rows = []
for n, d in J.items():
    flat = [x for v in d["labels"].values() for x in v]
    rows.append({"judge": n, "cost $": d["cost_usd"], "comparable": d["comparable"].get("ok", 0), **{l: flat.count(l) for l in LABELS}, "not comparable": flat.count(None)})
display(pd.DataFrame(rows).set_index("judge"))
def kappa(a, b):
    p = [(x, y) for x, y in zip(a, b) if x and y]
    if not p: return float("nan")
    po = sum(x == y for x, y in p) / len(p)
    pe = sum((sum(x == l for x, _ in p) / len(p)) * (sum(y == l for _, y in p) / len(p)) for l in LABELS)
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")
names = list(J); flat = {n: [x for w in WINDOWS for x in J[n]["labels"][w["id"]]] for n in names}
display(pd.DataFrame([[round(kappa(flat[a], flat[b]), 3) for b in names] for a in names], index=names, columns=names))